# 02 · Reasoning models — dial thinking up or down

Some Contoso Outdoors questions are one-liners ("what's in stock?"). Others need real reasoning — trip planning with weight budgets, weather risk, and money trade-offs all at once. Reasoning models let you dial how hard they think, and swap model size, independently of the prompt.

**Learning objectives**
- Tune `gpt-5.4`'s `reasoning_effort` (`low` → `high`) on a genuinely hard planning question
- See whether more effort beats a bigger model, or the other way around
- Compare `claude-sonnet-4-6` vs `claude-haiku-4-5` on the same hard question
- Walk away with a rule of thumb for when deeper reasoning is worth the cost

`~20 minutes`


## 1 · Set up and a helper to compare runs

Same `.env` as every other lab. `ask(model, prompt, **kwargs)` sends a prompt and returns the answer plus latency and token count — the two numbers reasoning effort trades against.


In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient


def find_agent_builder() -> Path:
    """Locate foundry/agent-builder from anywhere in the tree (repo root or labs/more)."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "foundry" / "agent-builder" / "src").is_dir():
            return base / "foundry" / "agent-builder"
        if base.name == "agent-builder" and (base / "src").is_dir():
            return base
    raise FileNotFoundError("Run labs/core/00-validate-setup.ipynb first — src/.env not found.")


AB = find_agent_builder()
load_dotenv(AB / "src" / ".env")
project_client = AIProjectClient(
    endpoint=os.environ["FOUNDRY_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()


def ask(model: str, prompt: str, **kwargs) -> dict:
    """Send one chat prompt and capture the answer plus cost/latency signals."""
    start = time.perf_counter()
    resp = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        **kwargs,
    )
    return {
        "asked_model": model,
        "served_model": resp.model,
        "latency_s": round(time.perf_counter() - start, 2),
        "total_tokens": getattr(resp.usage, "total_tokens", None),
        "answer": resp.choices[0].message.content,
    }


print("Ready. `ask(model, prompt, **kwargs)` returns latency, tokens, and the answer.")


## 2 · A genuinely hard question, two effort levels

A customer asks: plan gear and logistics for a 5-day backpacking trek, balancing a strict weight budget, uncertain weather, and a $600 gear budget — with trade-offs to justify, not just a list.

> ❓ **Does `reasoning_effort="high"` produce a genuinely better plan than `"low"`, or just a longer one?**


In [ ]:
HARD_TASK = (
    "Plan gear and daily logistics for a 5-day, 4-night backpacking trek in "
    "unpredictable shoulder-season weather (could be sunny or freezing rain). "
    "Total gear spend must stay under $600. Reason through the trade-offs "
    "between weight, warmth, and cost, and justify your final gear list."
)

effort_runs = []
for effort in ["low", "high"]:
    try:
        # reasoning_effort trades tokens/latency for deeper thinking.
        effort_runs.append({"effort": effort, **ask("gpt-5.4", HARD_TASK, reasoning_effort=effort)})
    except Exception as err:
        print(f"reasoning_effort='{effort}' not supported on this deployment ({err}).")

df = pd.DataFrame(effort_runs)
if not df.empty:
    df["answer_preview"] = df["answer"].str.replace("\n", " ").str.slice(0, 100) + "…"
df[["effort", "latency_s", "total_tokens", "answer_preview"]] if not df.empty else None


## 3 · Can effort make up for a smaller model?

`gpt-5.4-mini` is cheaper and faster than `gpt-5.4`. If we max out its reasoning effort, does it close the gap on the same hard question — or is model size a different lever entirely from effort?

> ❓ **Your call:** compare `total_tokens` and the plan quality. Would you trust the mini model's plan for a real trip?


In [ ]:
size_runs = []
for model in ["gpt-5.4-mini", "gpt-5.4"]:
    try:
        size_runs.append({"model": model, **ask(model, HARD_TASK, reasoning_effort="high")})
    except Exception as err:
        print(f"'{model}' at reasoning_effort='high' failed ({err}).")

df = pd.DataFrame(size_runs)
if not df.empty:
    df["answer_preview"] = df["answer"].str.replace("\n", " ").str.slice(0, 100) + "…"
df[["model", "latency_s", "total_tokens", "answer_preview"]] if not df.empty else None


## 4 · Optional — Claude on the same hard question

If `claude-sonnet-4-6` and `claude-haiku-4-5` are deployed (Skillable has both; self-guided is optional), we run the same trek-planning question on both and compare across vendors, not just across GPT sizes. If they're not deployed, this cell skips cleanly.


In [ ]:
from IPython.display import display

available = {
    getattr(d, "name", None) or getattr(d, "deployment_name", None)
    for d in project_client.deployments.list()
}
CLAUDE_MODELS = ["claude-sonnet-4-6", "claude-haiku-4-5"]

if all(name in available for name in CLAUDE_MODELS):
    claude_runs = [{"model": m, **ask(m, HARD_TASK)} for m in CLAUDE_MODELS]
    df = pd.DataFrame(claude_runs)
    df["answer_preview"] = df["answer"].str.replace("\n", " ").str.slice(0, 100) + "…"
    display(df[["model", "latency_s", "total_tokens", "answer_preview"]])
else:
    missing = [m for m in CLAUDE_MODELS if m not in available]
    print(f"Skipping — {', '.join(missing)} not deployed. Deploy with `provision.sh` to try this section.")


## 5 · When is deeper reasoning worth it?

| Signal you saw | Takeaway |
|---|---|
| `low` vs `high` effort on `gpt-5.4` | Higher effort costs more tokens/latency — worth it only if the answer is genuinely more useful, not just longer |
| `gpt-5.4-mini` at `high` effort vs `gpt-5.4` | Effort and model size are different levers — a bigger model isn't the same as a harder-thinking small one |
| Claude Sonnet vs Haiku (if deployed) | Same pattern across vendors: reach for the bigger model when the task has real trade-offs to weigh |

**Rule of thumb:** default to `low` effort and the smaller model. Reach for `high` effort or a bigger model only when the task has genuine ambiguity, competing constraints, or a wrong answer is costly.


## 🧭 Summary — so, when do you dial reasoning up?

You compared the same hard question across three levers — effort, model size, and vendor — and the answer was consistent: dial up only when the task earns it.

### Try it yourself
- Swap `HARD_TASK` for a simple factual question and see the effort levels converge — the lever matters less when the task doesn't need it.
- Try `reasoning_effort="medium"` as a middle ground.

### Next
➡️ **[03 · Image generation](03-image-generation.ipynb)** — generate and evaluate a Contoso Outdoors campaign poster with `MAI-Image-2.5-Pro`.
